# WRAP_P6_EXECUTION_SIM — Notebook 05  *(batch only)*

**Spec ref:** §9.3 — Statistical fill-probability + queue-priority analysis.

Single-order exploration lives in §10.2 UI tool, not here. Both share `fill_simulator`; this notebook calls `simulate_bulk`.

## Section 00 — Env & Imports

In [1]:
import sys, logging, random
from pathlib import Path
import numpy as np
import pandas as pd

# ── shared config ─────────────────────────────────────────────────────────────
_NB_DIR = Path('<repo>/p6lab/notebooks')
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))
import _common  # bootstraps sys.path; exposes NOTEBOOK_DATA_SLICE, helpers

from _common import (
    NOTEBOOK_DATA_SLICE, HORIZONS, PURGE_ROWS, TIER_CUTOFFS,
    BASELINE_MIN_AUC_IMPROVEMENT, collect_snapshots, collect_events, versioned_dir,
)
_DS       = NOTEBOOK_DATA_SLICE          # single source of truth
SYMBOL    = _DS['symbol']
TICK_SIZE = _DS['tick_size']

_P6LAB        = Path(_common._P6LAB_ROOT)
ARTIFACTS_DIR = _P6LAB / 'artifacts' / 'p6lab'
MAX_SNAPSHOTS = _DS['max_snapshots']

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)
np.random.seed(42)

print(f'Data  : {_DS["data_file"]}')
print(f'Symbol: {SYMBOL}  tick={TICK_SIZE}  max_snapshots={MAX_SNAPSHOTS}')
N_ORDERS = 200   # virtual orders per simulation run

from p6lab.execution.queue_tracker import QueueTracker, MatchingAlgorithm, Side
from p6lab.execution.fill_simulator import FillSimulator, OrderSpec, FillOutcome
from p6lab.execution.cost_model import CostModel, CostBreakdown

# To run a longer stress test, override here (e.g. 50_000 snapshots ≈ 80min,
# 2-min wall-clock). Leave at default for fast iteration.
# _DS = {**_DS, 'max_snapshots': 50_000}; MAX_SNAPSHOTS = _DS['max_snapshots']; N_ORDERS = 2_000


Data  : <repo>/data/nq-mbo-sample-15min.dbn.zst
Symbol: NQ  tick=▒.▒▒  max_snapshots=2000


## Section 01 — Replay with Queue Tracker

In [2]:
snaps = await collect_snapshots(_DS)
assert len(snaps) > 0, 'No snapshots collected — check _common.NOTEBOOK_DATA_SLICE["data_file"]'
log.info('01 │ %d snapshots loaded', len(snaps))

# Derive placement candidates: (timestamp_ms, best_bid, best_ask) per snapshot
placements = []
for s in snaps:
    if s.bids and s.asks:
        placements.append((s.timestamp_ms, s.bids[0].price, s.asks[0].price))
log.info('01 │ %d placement candidates', len(placements))

# --- MBO event stream (separate file pass, full event-id detail) ---
# Use the snapshot window's actual ts range so events overlap placements 1:1.
ts_lo = snaps[0].timestamp_ms
ts_hi = snaps[-1].timestamp_ms
mbo_slice = {**_DS, 'start_ms': ts_lo, 'end_ms': ts_hi}
events = await collect_events(mbo_slice)
log.info('01 │ %d MBO events loaded (window %d → %d ms)',
         len(events), ts_lo, ts_hi)
assert len(events) > 100, '01 │ too few MBO events — QueueTracker will starve'


Opening MBO data from <repo>/data/nq-mbo-sample-15min.dbn.zst (streaming mode)


Streaming ready. instrument_id=42004058


01 │ 2000 snapshots loaded


01 │ 2000 placement candidates


Opening MBO data from <repo>/data/nq-mbo-sample-15min.dbn.zst (streaming mode)


Streaming ready. instrument_id=42004058


01 │ 22008 MBO events loaded (window 1774476000101 → 1774476625726 ms)


## Section 02 — Signal Source

Real signals come from aggregator log + correlation engine; for the skeleton we sample uniform random buy/sell entries.

In [3]:
sampled = random.sample(placements, k=min(N_ORDERS, len(placements)))
orders = []
for ts, bid, ask in sampled:
    side, px = (Side.BUY, bid) if random.random() < 0.5 else (Side.SELL, ask)
    orders.append(OrderSpec(timestamp_ms=ts, side=side, price=px, size=1.0,
                            max_horizon_ms=10_000, adverse_exit_ticks=8))
orders.sort(key=lambda o: o.timestamp_ms)
log.info('02 │ %d signal orders', len(orders))


02 │ 200 signal orders


## Section 03 — Bulk Passive Simulation

In [4]:
sim = FillSimulator(tick_size=TICK_SIZE, tick_value=20.0)
outcomes = sim.simulate_bulk(orders, events)
n_filled = sum(1 for o in outcomes if o.filled)
log.info('03 │ fills %d/%d (%.0f%%)', n_filled, len(outcomes),
         100*n_filled/max(1,len(outcomes)))


03 │ fills 29/200 (14%)


## Section 04 — P(fill) by Queue-Position Quintile

For each virtual order, replay a parallel `QueueTracker` up to its placement time to capture its **entry queue position** (contracts ahead of us at that price level). Bucket into quintiles and report fill rate. Spec §6.1 claims P(fill) should decrease with queue position.

In [5]:
from p6lab.execution.queue_tracker import QueueTracker

# Parallel tracker: replay events up to each order's placement, capture queue position
tracker = QueueTracker(tick_size=TICK_SIZE)
pending = list(enumerate(orders))
entry_positions = [0.0] * len(orders)

ev_idx = 0
for order_idx, spec in pending:
    # Advance tracker until we pass this order's timestamp
    while ev_idx < len(events) and getattr(events[ev_idx], 'timestamp_ms', 0) <= spec.timestamp_ms:
        tracker.on_event(events[ev_idx])
        ev_idx += 1
    h = tracker.place_limit_order(spec.timestamp_ms, spec.side, spec.price, spec.size)
    pos = tracker.get_position(h)
    entry_positions[order_idx] = pos.position_from_front
    tracker.cancel_order(h)  # clean up — don't contaminate queue state

entry_arr = np.array(entry_positions)
log.info('04 │ entry position range: [%.1f, %.1f], median=%.1f',
         entry_arr.min(), entry_arr.max(), np.median(entry_arr))

# Quintile bucketing
if entry_arr.max() > entry_arr.min():
    edges = np.quantile(entry_arr, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
    rows = []
    for q in range(5):
        lo, hi = edges[q], edges[q + 1]
        mask = (entry_arr >= lo) & (entry_arr <= hi if q == 4 else entry_arr < hi)
        bucket = [outcomes[i] for i in np.where(mask)[0]]
        if not bucket:
            continue
        n_filled = sum(1 for o in bucket if o.filled)
        n_adverse = sum(1 for o in bucket if o.fill_reason == 'adverse_exit')
        rows.append({
            'quintile': f'Q{q}', 'range': f'[{lo:.0f}, {hi:.0f}]',
            'n': len(bucket), 'fill_rate': n_filled / len(bucket),
            'adverse_rate': n_adverse / len(bucket),
        })
    quintile_df = pd.DataFrame(rows)
    log.info('04 │ quintile results:\n' + quintile_df.to_string(index=False))
else:
    quintile_df = pd.DataFrame()
    log.info('04 │ all orders have identical queue position — quintiles collapsed')
quintile_df


04 │ entry position range: [▒.▒▒, ▒.▒▒], median=▒.▒▒


04 │ quintile results:
quintile   range  n  fill_rate  adverse_rate
      Q0  [0, 1] 24   ▒.▒▒      ▒.▒▒
      Q2  [1, 2] 86   ▒.▒▒      ▒.▒▒
      Q4 [2, 19] 90   ▒.▒▒      ▒.▒▒


,quintile,range,n,fill_rate,adverse_rate
0,Q0,"[0, 1]",24,▒.▒▒,▒.▒▒
1,Q2,"[1, 2]",86,▒.▒▒,▒.▒▒
2,Q4,"[2, 19]",90,▒.▒▒,▒.▒▒


In [6]:
# §04 assertion gate
assert len(outcomes) > 0, '§04 │ no fill outcomes produced'
_rate = sum(1 for o in outcomes if o.filled) / len(outcomes)
assert 0.0 <= _rate <= 1.0, f'§04 │ fill rate out of range: {_rate}'
print(f'✓ §04 gate PASS  overall fill={_rate:.1%}  quintiles={len(quintile_df)}')


✓ §04 gate PASS  overall fill=▒.▒▒%  quintiles=3


## Section 05 — Aggressive vs Passive

Simulate the same signal set as aggressive (cross-the-spread) entries at the opposing best quote. Aggressive always fills but pays half-spread; compare fill rates AND net P&L against the passive baseline.

In [7]:
# Aggressive orders: cross the spread via simulate_marketable.
# Each order is simulated independently against the event stream
# (stream is cheap to rescan at 600k events with a short walk-forward).
from p6lab.execution.fill_simulator import OrderSpec

sampled_agg = []
for spec, (ts, bid, ask) in zip(orders, random.sample(placements, k=len(orders))):
    if spec.side == Side.BUY:
        sampled_agg.append(OrderSpec(timestamp_ms=spec.timestamp_ms, side=Side.BUY,
                                     price=ask, size=1.0,
                                     max_horizon_ms=1_000, adverse_exit_ticks=8))
    else:
        sampled_agg.append(OrderSpec(timestamp_ms=spec.timestamp_ms, side=Side.SELL,
                                     price=bid, size=1.0,
                                     max_horizon_ms=1_000, adverse_exit_ticks=8))
sampled_agg.sort(key=lambda o: o.timestamp_ms)

# Walk the event stream once per order — use iter() so each marketable call
# starts from the same base events list. Cheap because we stop as soon as
# order.timestamp_ms is reached and the book is populated.
sim_agg = FillSimulator(tick_size=TICK_SIZE, tick_value=20.0)
outcomes_agg = [sim_agg.simulate_marketable(o, iter(events)) for o in sampled_agg]
n_filled_agg = sum(1 for o in outcomes_agg if o.filled)
n_filled_pas = sum(1 for o in outcomes if o.filled)

comparison = pd.DataFrame([
    {'entry_type': 'passive',    'n': len(outcomes),     'filled': n_filled_pas,
     'fill_rate': n_filled_pas / max(1, len(outcomes))},
    {'entry_type': 'aggressive', 'n': len(outcomes_agg), 'filled': n_filled_agg,
     'fill_rate': n_filled_agg / max(1, len(outcomes_agg))},
])
log.info('05 │ aggressive vs passive:\n' + comparison.to_string(index=False))
assert n_filled_agg / max(1, len(outcomes_agg)) > 0.8, (
    f'§05 │ aggressive fill rate {n_filled_agg}/{len(outcomes_agg)} '
    f'too low — simulate_marketable regression')
comparison


05 │ aggressive vs passive:
entry_type   n  filled  fill_rate
   passive 200      29      ▒.▒▒
aggressive 200     200      ▒.▒▒


,entry_type,n,filled,fill_rate
0,passive,200,29,▒.▒▒
1,aggressive,200,200,▒.▒▒


## Section 06 — Step-Ahead Economics

What if we placed orders **one tick further from touch** (deeper passive)? Simulate that alternative and compare fill rate vs additional edge gained.

In [8]:
# Same orders but 1 tick further from touch
sampled_step = []
for spec in orders:
    if spec.side == Side.BUY:
        new_px = spec.price - TICK_SIZE  # bid: go lower
    else:
        new_px = spec.price + TICK_SIZE  # ask: go higher
    sampled_step.append(OrderSpec(timestamp_ms=spec.timestamp_ms, side=spec.side,
                                  price=new_px, size=1.0,
                                  max_horizon_ms=10_000, adverse_exit_ticks=8))

sim_step = FillSimulator(tick_size=TICK_SIZE, tick_value=20.0)
outcomes_step = sim_step.simulate_bulk(sampled_step, events)
n_step_filled = sum(1 for o in outcomes_step if o.filled)

step_ahead = pd.DataFrame([
    {'placement': 'at touch',   'n_filled': n_filled_pas,  'fill_rate': n_filled_pas / len(orders),
     'edge_ticks_if_filled': 0},
    {'placement': '1 tick back', 'n_filled': n_step_filled, 'fill_rate': n_step_filled / len(orders),
     'edge_ticks_if_filled': 1},
])
log.info('06 │ step-ahead economics:\n' + step_ahead.to_string(index=False))
step_ahead


06 │ step-ahead economics:
  placement  n_filled  fill_rate  edge_ticks_if_filled
   at touch        29      ▒.▒▒                     0
1 tick back        31      ▒.▒▒                     1


,placement,n_filled,fill_rate,edge_ticks_if_filled
0,at touch,29,▒.▒▒,0
1,1 tick back,31,▒.▒▒,1


## Section 07 — Signal-to-Tradeable Conversion Rate

In [9]:
rate_passive = n_filled_pas / max(1, len(orders))
rate_agg = n_filled_agg / max(1, len(orders))
log.info('07 │ conversion: passive %.1f%%, aggressive %.1f%% (cost: +0.5 spread/fill)',
         100 * rate_passive, 100 * rate_agg)


07 │ conversion: passive ▒.▒▒%, aggressive ▒.▒▒% (cost: +▒.▒▒ spread/fill)


## Section 08 — Realistic PnL Backtest (4-component cost model)

For every **filled** order, compute PnL = filled_size × tick_value × (entry_edge_ticks − adverse_ticks), subtract full cost breakdown (crossed-spread / commission / adverse selection / opportunity).

In [10]:
from p6lab.execution.cost_model import CostModel
import math

cm_passive = CostModel(tick_value=20.0)
cm_aggressive = CostModel(tick_value=20.0, default_spread_ticks=1.0)

pnl_rows = []
for i, (spec, outcome) in enumerate(zip(orders, outcomes)):
    if not outcome.filled:
        continue
    # Gross PnL in $ = filled_size * tick_value * adverse_ticks_at_fill
    # (adverse ticks already captures how far mid moved against us before fill)
    adverse = getattr(outcome, 'adverse_ticks_at_fill', 0) or 0
    gross_ticks = -adverse  # negative because adverse
    gross_pnl = gross_ticks * 20.0 * outcome.filled_size
    costs = cm_passive.compute(outcome, entry_type='passive', spread_ticks=1.0)
    net_pnl = gross_pnl - costs.total_cost
    pnl_rows.append({
        'order_id': i, 'side': spec.side.name, 'entry_px': spec.price,
        'adverse_ticks': adverse, 'gross_pnl': gross_pnl,
        'cost': costs.total_cost, 'net_pnl': net_pnl,
    })
pnl_df = pd.DataFrame(pnl_rows)

summary = {
    'total_orders': len(orders),
    'filled': len(pnl_df),
    'fill_rate': len(pnl_df) / max(1, len(orders)),
    'gross_pnl_total': pnl_df['gross_pnl'].sum() if len(pnl_df) else 0.0,
    'cost_total': pnl_df['cost'].sum() if len(pnl_df) else 0.0,
    'net_pnl_total': pnl_df['net_pnl'].sum() if len(pnl_df) else 0.0,
    'net_pnl_per_order': pnl_df['net_pnl'].mean() if len(pnl_df) else 0.0,
}
log.info('08 │ realistic PnL: gross=$%.2f costs=$%.2f NET=$%.2f across %d fills',
         summary['gross_pnl_total'], summary['cost_total'],
         summary['net_pnl_total'], summary['filled'])
log.info('08 │ per-order net: $%.3f (avg)', summary['net_pnl_per_order'])

# Naive comparison
breakdowns = cm_passive.compute_batch(outcomes)
naive_cmp = cm_passive.compare_to_naive(breakdowns)
log.info('08 │ realistic mean cost $%.2f vs naive $%.2f (ratio %.2fx)',
         naive_cmp['realistic_mean'], naive_cmp['naive_mean'], naive_cmp['cost_ratio'])
pnl_df.head(10)


08 │ realistic PnL: gross=$▒.▒▒ costs=$▒.▒▒ NET=$▒.▒▒ across 29 fills


08 │ per-order net: $▒.▒▒ (avg)


08 │ realistic mean cost $▒.▒▒ vs naive $▒.▒▒ (ratio ▒.▒▒x)


,order_id,side,entry_px,adverse_ticks,gross_pnl,cost,net_pnl
0,3,BUY,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
1,4,BUY,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
2,9,BUY,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
3,10,SELL,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
4,14,BUY,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
5,16,SELL,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
6,18,SELL,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
7,19,BUY,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
8,20,SELL,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒
9,24,SELL,▒.▒▒,0,▒.▒▒,▒.▒▒,▒.▒▒


## Section 09 — Report

Writes `artifacts/p6lab/execution/nb05_report.md`.

In [11]:
report_dir = versioned_dir(ARTIFACTS_DIR / 'execution', 'nb05', data_slice=_DS, extra={'n_orders': N_ORDERS, 'fill_rate': float(_rate), 'cost_ratio': float(realistic_mean / max(naive_mean, 1e-9)) if 'realistic_mean' in dir() else None})
report_path = report_dir / 'nb05_report.md'

with open(report_path, 'w') as f:
    f.write(f'# NB05 — Execution Sim Report\n\n')
    f.write(f'**Symbol:** {SYMBOL}  \n')
    f.write(f'**Orders:** {len(orders)}  **Events:** {len(events):,}  \n')
    f.write(f'**Passive fill rate:** {rate_passive:.1%}  \n')
    f.write(f'**Aggressive fill rate:** {rate_agg:.1%}  \n\n')
    f.write('## Queue-Position Quintile P(fill)\n\n')
    if len(quintile_df):
        f.write(quintile_df.to_markdown(index=False))
        f.write('\n\n')
    f.write('## Aggressive vs Passive\n\n')
    f.write(comparison.to_markdown(index=False))
    f.write('\n\n## Step-Ahead Economics\n\n')
    f.write(step_ahead.to_markdown(index=False))
    f.write('\n\n## Realistic PnL\n\n')
    f.write(f'- Filled: **{summary["filled"]}/{summary["total_orders"]}**\n')
    f.write(f'- Gross PnL: **${summary["gross_pnl_total"]:+.2f}**\n')
    f.write(f'- Total costs: **${summary["cost_total"]:.2f}**\n')
    f.write(f'- **Net PnL: ${summary["net_pnl_total"]:+.2f}**\n')
    f.write(f'- Per-order net: **${summary["net_pnl_per_order"]:+.3f}**\n\n')
    f.write(f'## Cost Comparison\n\n')
    f.write(f'- Realistic mean: ${naive_cmp["realistic_mean"]:.2f}/contract\n')
    f.write(f'- Naive mean:     ${naive_cmp["naive_mean"]:.2f}/contract\n')
    f.write(f'- Ratio: **{naive_cmp["cost_ratio"]:.2f}×**\n')
    f.write(f'- Adverse-selection pct: {naive_cmp["adverse_selection_pct"]*100:.1f}%\n')
log.info('09 │ wrote %s (%.1f KB)', report_path, report_path.stat().st_size / 1024)


09 │ wrote <repo>/p6lab/artifacts/p6lab/execution/nb05_20260420_1648/nb05_report.md (▒.▒▒ KB)
